# RQ1 reproducibility notebook

Reproduces the **Dataset** subsection and **RQ1 (availability, length, readability)** results reported in `Findings.tex`. Every number printed here lines up with a number or figure quoted in the paper.

Flow:

1. Set up paths, extract the bundled crawl output.
2. Dataset overview (universe sizes, scale).
3. RQ1 funnel and Table 4 (per-role availability and length).
4. Figure 4(a): first-party availability by Tranco rank.
5. Figure 4(b): first-party availability by content category.
6. Figure 5: third-party availability by prevalence rank.
7. Length and readability statistics.
8. **Additional**: extra metrics and graphs not directly cited in the paper (TP density, length histograms, FP × TP service heat-map, top companies).


## 1. Setup


### 1.1 Decompress the bundled dataset


In [ ]:
import os, tarfile, pathlib

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
DATA_DIR  = REPO_ROOT / 'data' / 'raw'
BUNDLE    = REPO_ROOT / 'data' / 'dataset.tar.gz'

DATA_DIR.mkdir(parents=True, exist_ok=True)
if (DATA_DIR / 'results.jsonl').exists():
    print(f'Already extracted at {DATA_DIR}')
else:
    assert BUNDLE.exists(), f'Missing {BUNDLE}.'
    print(f'Extracting {BUNDLE.name} ...')
    with tarfile.open(BUNDLE, 'r:gz') as tf:
        tf.extractall(DATA_DIR)
    print('Done.')


### 1.2 Imports and the canonical filter

A *qualifying* policy is, throughout the paper, an English policy of at least 500 words whose URL is not on the curation blacklist. This is the same filter the figure-rendering scripts use, so per-bucket percentages below match the figures in the paper exactly.


In [ ]:
import json, re, glob, collections
import numpy as np
import matplotlib.pyplot as plt

MIN_WORDS = 500
WORD_RE   = re.compile(r'\S+')


def cache_wc(entry):
    # Word count of a cached TP policy text, restricted to English.
    # Returns None for non-English entries (treated as no text downstream).
    if not isinstance(entry, dict):
        return None
    if entry.get('language', 'en') != 'en':
        return None
    wc = entry.get('word_count')
    if wc is None:
        wc = len(WORD_RE.findall(entry.get('text') or ''))
    return wc


def percentile(xs, p):
    xs = sorted(xs)
    if not xs:
        return 0
    k = (len(xs) - 1) * p / 100
    f = int(k); c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)


### 1.3 Load the bundled crawl output

Loads:
- `results.jsonl` — one record per first-party site we crawled.
- The third-party policy text caches.
- `policy_curation.json` — the official blacklists (first-party eTLD+1s and third-party policy URLs).


In [ ]:
rows = [json.loads(ln) for ln in open(DATA_DIR / 'results.jsonl', encoding='utf-8')]
print(f'Records in results.jsonl: {len(rows):,}')

tp_cache = {}
for f in sorted(glob.glob(str(DATA_DIR / 'results_shard*.tp_cache.json'))):
    tp_cache.update(json.load(open(f, encoding='utf-8')))
print(f'Cached third-party policy URLs: {len(tp_cache):,}')

# Some third parties had a missing policy URL on the first pass and were
# re-discovered later. This dict maps the third-party domain to the URL
# the rediscovery pass found.
redisc = {e['rediscovered_from_etld1'].lower(): u
          for u, e in tp_cache.items()
          if isinstance(e, dict) and e.get('rediscovered_from_etld1')}
print(f'Rediscovered policy URLs: {len(redisc):,}')

curation = json.load(open(DATA_DIR / 'policy_curation.json', encoding='utf-8'))
fp_blacklist = set(curation['fp_blacklist_etld1'])
tp_blacklist = set(curation['tp_blacklist_urls'])
print(f'FP blacklist size: {len(fp_blacklist):,}')
print(f'TP blacklist size: {len(tp_blacklist):,}')


## 2. Dataset

Reproduces the *Dataset* paragraph in `Findings.tex`.


### 2.1 Funnel from the Tranco list to the crawl universe


In [ ]:
summary = json.load(open(DATA_DIR / 'results.summary.json', encoding='utf-8'))

TRANCO_SEED = 16_100
AFTER_CRUX  = 7_489            # number reported in the paper
processed   = summary['processed_sites']
home_ok     = summary['home_ok_count']
status      = summary.get('status_counts', {})

print('Tranco top-16,100 ............................. 16,100')
print(f'After Chrome UX Report filter ................. {AFTER_CRUX:>6,}')
print(f'Crawled (full attempt) ........................ {processed:>6,}')
print(f'Homepage loaded successfully .................. {home_ok:>6,}')
print(f'  with policy link found ...................... {status.get("ok", 0):>6,}')
print(f'  without policy link ......................... {status.get("policy_not_found", 0):>6,}')
fail = (status.get('non_browsable', 0)
        + status.get('home_fetch_failed', 0)
        + status.get('exception', 0))
print(f'Failed to load (sum) .......................... {fail:>6,}')
print(f'  non-browsable ............................... {status.get("non_browsable", 0):>6,}')
print(f'  home-fetch failed ........................... {status.get("home_fetch_failed", 0):>6,}')
print(f'  rendering exception ......................... {status.get("exception", 0):>6,}')


### 2.2 Distinct third-party domains observed


In [ ]:
tp_observed = set()
for r in rows:
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict):
            continue
        et = (tp.get('third_party_etld1') or '').lower()
        if et:
            tp_observed.add(et)

print(f'Distinct third-party domains observed: {len(tp_observed):,}')


### 2.3 Scale: total words of policy text

The collection holds roughly 21.5 million words of privacy-policy text, split between first-party documents (17.2 M) and third-party documents (4.3 M).


In [ ]:
# First-party words: sum over all first parties whose policy qualifies
fp_qual_rows = [
    r for r in rows
    if r.get('status') == 'ok'
    and r.get('policy_is_english')
    and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
    and (r.get('site_etld1') or '').lower() not in fp_blacklist
]
fp_words = sum(r['first_party_policy_word_count'] for r in fp_qual_rows)
print(f'Qualifying first-party policies: {len(fp_qual_rows):,}')
print(f'First-party words total: {fp_words:>12,} ({fp_words/1e6:>4.1f} M)')

# Third-party words: sum over the unique documents that are actually
# associated with observed qualifying third parties (a TP cache entry that
# nobody references in the crawl is irrelevant to the dataset).
qual_urls_all = {u for u, e in tp_cache.items()
                 if (cache_wc(e) or 0) >= MIN_WORDS and u not in tp_blacklist}

tp_url_used = set()
for r in rows:
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        et = (tp.get('third_party_etld1') or '').lower()
        if not et: continue
        url = tp.get('policy_url') or redisc.get(et)
        if url and url in qual_urls_all:
            tp_url_used.add(url)

tp_words = sum(cache_wc(tp_cache[u]) for u in tp_url_used)
print(f'Unique third-party documents actually used: {len(tp_url_used):,}')
print(f'Third-party words total: {tp_words:>12,} ({tp_words/1e6:>4.1f} M)')

print(f'\nGrand total: {fp_words + tp_words:>12,} ({(fp_words + tp_words)/1e6:>4.1f} M)')

# Re-export the smaller URL set as `qual_urls` for downstream cells
qual_urls = qual_urls_all


## 3. RQ1: Availability, length, and readability

Reproduces Table 4 and the three RQ1 figures in `Findings.tex`.


### 3.1 Per-role availability (Table 4)


In [ ]:
# First-party column
fp_observed_n = len(rows)
fp_url_known  = sum(1 for r in rows if r.get('status') == 'ok')
fp_qualifying = len(fp_qual_rows)

# Third-party column: domains observed across the home_ok crawl
tp_url_known = set()
tp_qual      = set()
for r in rows:
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict):
            continue
        et = (tp.get('third_party_etld1') or '').lower()
        if not et:
            continue
        url = tp.get('policy_url') or redisc.get(et)
        if url:
            tp_url_known.add(et)
            if url in qual_urls:
                tp_qual.add(et)

tp_observed_n = len(tp_observed)
tp_url_known_n = len(tp_url_known)
tp_qualifying = len(tp_qual)

def fmt(num, den):
    return f'{100*num/den:5.1f}%' if den else '   --'

print('                                  First party   Third party')
print('-' * 60)
print(f'Observed domains                  {fp_observed_n:>10,}    {tp_observed_n:>10,}')
print(f'Policy URL known                  {fp_url_known:>10,}    {tp_url_known_n:>10,}')
print(f'Qualifying policy                 {fp_qualifying:>10,}    {tp_qualifying:>10,}')
print()
print(f'Availability                      {fmt(fp_qualifying, fp_observed_n):>10}    {fmt(tp_qualifying, tp_observed_n):>10}')
print(f'Text retrieved (when URL known)   {fmt(fp_qualifying, fp_url_known):>10}    {fmt(tp_qualifying, tp_url_known_n):>10}')


### 3.2 Figure 4(a): availability by Tranco rank

Each bucket reports the share of *home-rendering* first-party sites in that rank range that publish a qualifying English policy.


In [ ]:
records = []
for r in rows:
    if r.get('home_ok') is not True:
        continue
    rank = r.get('rank')
    if not isinstance(rank, int):
        continue
    et = (r.get('site_etld1') or '').lower()
    qualifies = (
        r.get('status') == 'ok'
        and r.get('policy_is_english')
        and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
        and et not in fp_blacklist
    )
    records.append((rank, 1 if qualifies else 0))

records.sort()
ranks = np.array([r for r, _ in records])
qual  = np.array([q for _, q in records])

buckets = [(1,100),(101,500),(501,1000),(1001,2500),(2501,5000),(5001,10000),(10001,int(ranks.max()))]
labels, avail = [], []
for lo, hi in buckets:
    mask = (ranks >= lo) & (ranks <= hi)
    if mask.sum() == 0:
        continue
    labels.append(f'{lo:,}-{hi:,}')
    avail.append(100 * qual[mask].sum() / mask.sum())
    print(f'{labels[-1]:<14s} {avail[-1]:>5.1f}%   ({int(qual[mask].sum())} of {int(mask.sum())} sites)')

PRIMARY = '#2171b5'
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'pdf.fonttype': 42,
})
fig, ax = plt.subplots(figsize=(3.35, 2.5))
y = np.arange(len(labels))[::-1]
cov = np.array(avail); unc = 100 - cov
ax.barh(y, cov, height=0.7, color=PRIMARY, edgecolor='white', linewidth=0.5, label='Policy available')
ax.barh(y, unc, left=cov, height=0.7, color='#d9d9d9', edgecolor='white', linewidth=0.5, label='Policy not available')
for yi, p in zip(y, avail):
    if p >= 18:
        ax.text(p/2, yi, f'{p:.0f}%', ha='center', va='center', color='white', fontsize=7, fontweight='bold')
    else:
        ax.text(p+1.5, yi, f'{p:.0f}%', ha='left', va='center', color='#333333', fontsize=7, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8)
ax.set_xlim(0, 102); ax.set_xticks([0, 25, 50, 75, 100])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%' if x <= 100 else ''))
ax.set_ylabel('Tranco rank', fontsize=9)
ax.tick_params(axis='x', labelsize=8); ax.tick_params(axis='y', pad=2)
ax.grid(True, axis='x', alpha=0.25, linestyle=':', linewidth=0.5); ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=7, frameon=False, handlelength=1.2, handletextpad=0.4, columnspacing=1.0)
plt.tight_layout(pad=0.3); plt.show()


### 3.3 Figure 4(b): availability by content category

Same denominator as 3.2 (home-rendering first parties), bucketed by content category. The figure uses short labels — full names are in the printed table.


In [ ]:
ABBREV = {
    'Nonprofit & Religion':    'Nonprofit',
    'Social & Communication':  'Social',
    'Technology':              'Tech',
    'Web Infrastructure':      'Web Infra',
    'Business & Finance':      'Business',
    'Entertainment':           'Entert.',
    'E-commerce':              'E-comm.',
    'Education':               'Edu.',
    'News & Media':            'News',
    'Government':              "Gov't",
    'Security Risks':          'Security',
}

per_cat_total = collections.Counter()
per_cat_qual  = collections.Counter()
for r in rows:
    if r.get('home_ok') is not True:
        continue
    cat = r.get('main_category') or 'Uncategorized'
    per_cat_total[cat] += 1
    et = (r.get('site_etld1') or '').lower()
    qualifies = (
        r.get('status') == 'ok'
        and r.get('policy_is_english')
        and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
        and et not in fp_blacklist
    )
    if qualifies:
        per_cat_qual[cat] += 1

cats  = [c for c in per_cat_total if per_cat_total[c] >= 5]
avail = [100 * per_cat_qual[c] / per_cat_total[c] for c in cats]
order = np.argsort(avail)[::-1]
cats_s  = [cats[i] for i in order]
avail_s = [avail[i] for i in order]

print(f'{"Category":<24s} {"Avail":>8s} {"Qual":>6s} {"Total":>6s}')
print('-' * 50)
for c, a in zip(cats_s, avail_s):
    print(f'{c:<24s} {a:>7.1f}%  {per_cat_qual[c]:>5,}  {per_cat_total[c]:>5,}')

labels = [ABBREV.get(c, c) for c in cats_s]
fig, ax = plt.subplots(figsize=(3.35, 2.6))
x = np.arange(len(cats_s))
ax.bar(x, avail_s, color=PRIMARY, edgecolor='#0b3d6b', linewidth=0.5, width=0.72)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=7, rotation=45, ha='right', rotation_mode='anchor')
ax.set_ylabel('Share with qualifying policy', fontsize=7.5)
ax.set_ylim(0, 100); ax.set_yticks([0, 25, 50, 75, 100])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.tick_params(axis='y', labelsize=7); ax.tick_params(axis='x', pad=1)
ax.grid(True, axis='y', alpha=0.25, linestyle=':', linewidth=0.5); ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(pad=0.3); plt.show()


### 3.4 Figure 5: third-party availability by prevalence rank

Third parties are ordered by *prevalence* — the number of distinct first-party sites that contact the third party at load time.


In [ ]:
# Build the "reachable" set from the rediscovery log + index hits.
tp_home_reachable = set()
rfile = DATA_DIR / 'tp_rediscovery_full.jsonl'
if rfile.exists():
    bad = {'home_unreachable', 'fetch_error'}
    for ln in open(rfile, encoding='utf-8'):
        rec = json.loads(ln)
        d = (rec.get('domain') or rec.get('etld1') or '').lower()
        if d and rec.get('outcome') and rec['outcome'] not in bad:
            tp_home_reachable.add(d)

tp_obs_counter = collections.Counter()
tp_covered, tp_in_index = set(), set()
for r in rows:
    if r.get('home_ok') is not True:
        continue
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        et = (tp.get('third_party_etld1') or '').lower()
        if not et: continue
        tp_obs_counter[et] += 1
        if (tp.get('tracker_radar_source_domain_file')
                or tp.get('trackerdb_source_pattern_file')
                or tp.get('trackerdb_source_org_file')):
            tp_in_index.add(et)
        url = tp.get('policy_url') or redisc.get(et)
        if url and url in qual_urls:
            tp_covered.add(et)

reachable = tp_in_index | tp_home_reachable | tp_covered
tp_obs_counter = collections.Counter({d: n for d, n in tp_obs_counter.items() if d in reachable})
tp_covered = {d for d in tp_covered if d in reachable}
ranked = sorted(tp_obs_counter.items(), key=lambda x: -x[1])
n = len(ranked)
covered_flag = np.array([1 if d in tp_covered else 0 for d, _ in ranked])

buckets = [(1,100),(101,250),(251,500),(501,1000),(1001,2000),(2001,n)]
labels, avail = [], []
for lo, hi in buckets:
    if lo > n:
        continue
    hi = min(hi, n)
    seg = covered_flag[lo-1:hi]
    labels.append(f'{lo:,}-{hi:,}')
    avail.append(100 * seg.sum() / len(seg))
    print(f'{labels[-1]:<14s} {avail[-1]:>5.1f}%   ({int(seg.sum())} of {len(seg)} third parties)')

fig, ax = plt.subplots(figsize=(7.0, 3.0))
y = np.arange(len(labels))[::-1]
cov = np.array(avail); unc = 100 - cov
ax.barh(y, cov, height=0.65, color=PRIMARY, edgecolor='white', linewidth=0.6, label='Policy available')
ax.barh(y, unc, left=cov, height=0.65, color='#d9d9d9', edgecolor='white', linewidth=0.6, label='Policy not available')
for yi, p in zip(y, avail):
    if p >= 20:
        ax.text(p/2, yi, f'{p:.0f}%', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
    else:
        ax.text(p+1.2, yi, f'{p:.0f}%', ha='left', va='center', color='#333333', fontsize=9, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlim(0, 102); ax.set_xticks([0, 25, 50, 75, 100])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%' if x <= 100 else ''))
ax.set_ylabel('TP prevalence rank', fontsize=10)
ax.grid(True, axis='x', alpha=0.22, linestyle=':'); ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=9, frameon=False)
plt.tight_layout(); plt.show()


### 3.5 Length and readability

When text is available, first-party and third-party policies sit in the same length range — medians differ by ~550 words, IQRs overlap broadly.


In [ ]:
fp_wcs = [r['first_party_policy_word_count'] for r in fp_qual_rows]
# Third-party length stats use the unique documents actually used by the
# crawl (the 870 unique documents behind the 1,122 qualifying TP domains
# referenced in the paper's Table 4 caption).
tp_wcs = [cache_wc(tp_cache[u]) for u in tp_url_used]

def describe(label, xs):
    print(f'{label:<14s} n={len(xs):>5,}   '
          f'median={percentile(xs,50):>6,.0f}   '
          f'mean={sum(xs)/len(xs):>6,.0f}   '
          f'IQR={percentile(xs,25):>5,.0f}-{percentile(xs,75):<5,.0f}')

describe('First-party', fp_wcs)
describe('Third-party', tp_wcs)
print(f'\nUnique third-party documents behind {len(tp_qual):,} qualifying domains: {len(tp_url_used):,}')


## 4. Additional metrics and graphs

Beyond the numbers reported in `Findings.tex`. Useful for understanding the dataset; not required for paper reproduction.


### 4.1 Word-count distributions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.5, 2.6))
for ax, xs, title in zip(axes, [fp_wcs, tp_wcs], ['First-party policies', 'Third-party policies']):
    bins = np.linspace(0, 20000, 41)
    ax.hist(np.clip(xs, 0, 20000), bins=bins, color=PRIMARY, edgecolor='white', linewidth=0.4)
    ax.axvline(percentile(xs, 50), color='#d62728', linestyle='--', linewidth=1, label=f'median {percentile(xs,50):.0f}')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Word count', fontsize=9)
    ax.set_xlim(0, 20000)
    ax.tick_params(axis='both', labelsize=8)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.legend(fontsize=8, frameon=False)
axes[0].set_ylabel('Number of policies', fontsize=9)
plt.tight_layout(pad=0.5); plt.show()
print(f'First-party policies above 20K words (clipped): {sum(1 for w in fp_wcs if w > 20000):,}')
print(f'Third-party policies above 20K words (clipped): {sum(1 for w in tp_wcs if w > 20000):,}')


### 4.2 Third-party density per qualifying first party


In [ ]:
qualifying_fp_set = {r['site_etld1'].lower() for r in fp_qual_rows}
qualifying_sites = set()
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualifying_fp_set:
        continue
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        et = (tp.get('third_party_etld1') or '').lower()
        if not et: continue
        url = tp.get('policy_url') or redisc.get(et)
        if url in qual_urls:
            qualifying_sites.add(site_et)
            break

per_site_density = collections.Counter()
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualifying_sites: continue
    seen = set()
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        et = (tp.get('third_party_etld1') or '').lower()
        if not et: continue
        url = tp.get('policy_url') or redisc.get(et)
        if url in qual_urls and et not in seen:
            per_site_density[site_et] += 1
            seen.add(et)

vals = sorted(per_site_density.values())
print(f'Qualifying sites (FP + at least one qualifying TP): {len(qualifying_sites):,}')
print(f'Median qualifying TPs per site: {percentile(vals, 50):.0f}')
print(f'Mean: {sum(vals)/len(vals):.1f}   p90: {percentile(vals, 90):.0f}   max: {max(vals)}')

fig, ax = plt.subplots(figsize=(4.5, 2.6))
ax.hist(np.clip(vals, 0, 60), bins=np.arange(0, 62, 2), color=PRIMARY, edgecolor='white', linewidth=0.4)
ax.axvline(percentile(vals, 50), color='#d62728', linestyle='--', linewidth=1, label=f'median {percentile(vals,50):.0f}')
ax.set_xlabel('Qualifying third parties per site', fontsize=9)
ax.set_ylabel('Sites', fontsize=9)
ax.tick_params(axis='both', labelsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()


### 4.3 Heat-map: FP content category × TP service category

Joint view of which kinds of first parties embed which kinds of third parties. Each row sums to 100% (share of qualifying-pair appearances from FPs of that category that go to each TP service bucket).


In [ ]:
TP_BUCKETS = {
    'Advertising':         {'Advertising', 'Ad Motivated Tracking', 'Ad Fraud', 'Action Pixels'},
    'Analytics':           {'Analytics', 'Audience Measurement'},
    'CDN & Hosting':       {'CDN', 'Hosting', 'Content Delivery'},
    'Embedded Content':    {'Embedded Content', 'Federated Login'},
    'Tag Management':      {'Tag Manager'},
    'Consent Management':  {'Consent Management'},
    'Identity & Payment':  {'Identity', 'Online Payment'},
    'Social Media':        {'Social Network', 'Social', 'Social Media'},
    'High Risk':           {'Surveillance', 'Adult Advertising'},
}

def bucket_for(cats):
    for name, members in TP_BUCKETS.items():
        if any(c in members for c in (cats or [])):
            return name
    return 'Other'

heatmap = collections.Counter()
fp_totals = collections.Counter()
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualifying_sites:
        continue
    fp_cat = r.get('main_category') or 'Other'
    fp_totals[fp_cat] += 1
    seen = set()
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        et = (tp.get('third_party_etld1') or '').lower()
        if not et: continue
        url = tp.get('policy_url') or redisc.get(et)
        if url not in qual_urls: continue
        b = bucket_for(tp.get('categories'))
        if (et, b) in seen: continue
        seen.add((et, b))
        heatmap[(fp_cat, b)] += 1

fp_order = sorted(fp_totals, key=lambda c: -fp_totals[c])
tp_order = list(TP_BUCKETS.keys()) + ['Other']
M = np.zeros((len(fp_order), len(tp_order)))
for (fp, tb), n in heatmap.items():
    if fp not in fp_order or tb not in tp_order: continue
    i = fp_order.index(fp); j = tp_order.index(tb)
    M[i, j] = n

# Row-normalise so each row sums to 100% (share of pairs from that FP
# category that go to each TP bucket)
M_norm = M / np.where(M.sum(axis=1, keepdims=True) > 0, M.sum(axis=1, keepdims=True), 1) * 100

fig, ax = plt.subplots(figsize=(7.5, 4.5))
im = ax.imshow(M_norm, cmap='Blues', aspect='auto', vmin=0, vmax=M_norm.max())
ax.set_xticks(range(len(tp_order))); ax.set_xticklabels(tp_order, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(fp_order))); ax.set_yticklabels(fp_order, fontsize=8)
for i in range(M_norm.shape[0]):
    for j in range(M_norm.shape[1]):
        if M_norm[i, j] >= 1:
            ax.text(j, i, f'{M_norm[i,j]:.0f}%', ha='center', va='center', fontsize=7,
                    color='white' if M_norm[i,j] > M_norm.max()*0.55 else '#333333')
fig.colorbar(im, ax=ax, label='Row share (%)', shrink=0.8)
plt.tight_layout(); plt.show()


### 4.4 Top third-party companies on qualifying sites


In [ ]:
entity_counts = collections.Counter()
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualifying_sites: continue
    seen = set()
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        et = (tp.get('third_party_etld1') or '').lower()
        if not et or et in seen: continue
        seen.add(et)
        ent = tp.get('entity')
        if ent:
            entity_counts[ent] += 1

print(f'Distinct third-party companies on qualifying sites: {len(entity_counts):,}\n')
print(f'{"Rank":>4s}  {"Company":<40s}  {"Sites":>6s}  Share')
print('-' * 70)
total = sum(entity_counts.values())
for i, (ent, n) in enumerate(entity_counts.most_common(15), 1):
    print(f'{i:>4}.  {ent:<40s}  {n:>6,}  {100*n/total:5.1f}%')
